# Воспроизведение статьи об определении занятости помещения

**Статья:** Luis M. Candanedo, Véronique Feldheim, *Accurate occupancy detection of an office room from light, temperature, humidity and CO2 measurements using statistical learning models*, Energy and Buildings, 2016.  
**DOI:** https://doi.org/10.1016/j.enbuild.2015.11.071  
**Датасет:** https://archive.ics.uci.edu/dataset/357/occupancy+detection  
**Код авторов:** https://github.com/LuisM78/Occupancy-detection-data

Цель работы — воспроизвести один конкретный эксперимент статьи: Random Forest на всех пяти измеряемых признаках и двух признаках, извлечённых из времени. Сравнение проводится отдельно на двух опубликованных тестовых наборах, как у авторов.

## 1. Спецификация эксперимента

Измеряемые признаки: `Temperature`, `Humidity`, `Light`, `CO2`, `HumidityRatio`.

Обработка даты повторяет авторский R-скрипт:

- `NSM` — число секунд от начала суток;
- `WeekStatus` — рабочий день или выходной;
- исходная строка `date` в модель не передаётся;
- пропусков в данных нет, масштабирование для деревьев не применяется.

Random Forest: 500 деревьев, бутстрэп, критерий Gini, деревья без ограничения глубины, `mtry = 2`, `seed = 1234`. В `scikit-learn` параметру `mtry` соответствует `max_features`.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 3.0.3
scikit-learn: 1.8.0


## 2. Загрузка опубликованных выборок

Архив UCI хранится рядом с ноутбуком. Если его нет, код скачивает официальную копию. Индекс в первом столбце не является признаком.

In [2]:
DATA_URL = "https://archive.ics.uci.edu/static/public/357/occupancy+detection.zip"
DATA_DIR = Path("data")
ARCHIVE_PATH = DATA_DIR / "occupancy-detection.zip"

DATA_DIR.mkdir(exist_ok=True)
if not ARCHIVE_PATH.exists():
    urlretrieve(DATA_URL, ARCHIVE_PATH)

with ZipFile(ARCHIVE_PATH) as archive:
    members = {Path(name).name: name for name in archive.namelist()}

    def read_split(filename):
        with archive.open(members[filename]) as file:
            return pd.read_csv(file, index_col=0)

    train = read_split("datatraining.txt")
    test_1 = read_split("datatest.txt")
    test_2 = read_split("datatest2.txt")

summary = pd.DataFrame(
    {
        "objects": [len(train), len(test_1), len(test_2)],
        "columns": [train.shape[1], test_1.shape[1], test_2.shape[1]],
        "missing": [train.isna().sum().sum(), test_1.isna().sum().sum(), test_2.isna().sum().sum()],
        "occupied_share": [train["Occupancy"].mean(), test_1["Occupancy"].mean(), test_2["Occupancy"].mean()],
    },
    index=["train", "test_1", "test_2"],
)
summary

,objects,columns,missing,occupied_share
train,8143,7,0,0.212330
test_1,2665,7,0,0.364728
test_2,9752,7,0,0.210111


Размеры должны совпасть с описанием датасета: 8 143 объекта для обучения, 2 665 и 9 752 объекта в тестах, всего 20 560 наблюдений.

In [3]:
assert len(train) + len(test_1) + len(test_2) == 20_560
assert train.isna().sum().sum() + test_1.isna().sum().sum() + test_2.isna().sum().sum() == 0

SENSOR_FEATURES = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
MODEL_FEATURES = SENSOR_FEATURES + ["NSM", "WeekStatus"]

def prepare_data(frame):
    prepared = frame.copy()
    timestamp = pd.to_datetime(prepared["date"], utc=True)
    prepared["NSM"] = (
        timestamp.dt.hour * 3600
        + timestamp.dt.minute * 60
        + timestamp.dt.second
    )
    prepared["WeekStatus"] = (timestamp.dt.dayofweek < 5).astype(int)
    return prepared[MODEL_FEATURES], prepared["Occupancy"].astype(int)

X_train, y_train = prepare_data(train)
X_test_1, y_test_1 = prepare_data(test_1)
X_test_2, y_test_2 = prepare_data(test_2)

X_train.head()

,Temperature,Humidity,Light,CO2,HumidityRatio,NSM,WeekStatus
1,23.180000,27.272000,426.000000,721.250000,0.004793,64260,1
2,23.150000,27.267500,429.500000,714.000000,0.004783,64319,1
3,23.150000,27.245000,426.000000,713.500000,0.004779,64380,1
4,23.150000,27.200000,426.000000,708.250000,0.004772,64440,1
5,23.100000,27.200000,426.000000,704.500000,0.004757,64500,1


## 3. Обучение Random Forest

В авторском коде используется пакет R `randomForest` через `caret`. Его значения по умолчанию для этого эксперимента дают 500 деревьев и два случайно рассматриваемых признака на каждом разбиении. Остальные параметры ниже выбраны как наиболее близкое соответствие в `scikit-learn`.

In [4]:
model = RandomForestClassifier(
    n_estimators=500,
    criterion="gini",
    max_features=2,
    bootstrap=True,
    max_depth=None,
    min_samples_leaf=1,
    oob_score=True,
    random_state=1234,
    n_jobs=-1,
)
model.fit(X_train, y_train)

print(f"Train accuracy: {model.score(X_train, y_train):.6f}")
print(f"OOB accuracy:   {model.oob_score_:.6f}")

Train accuracy: 1.000000
OOB accuracy:   0.995456


## 4. Метрики на двух тестах

Авторы сравнивают модели прежде всего по accuracy. Дополнительно посчитаем precision, recall, specificity, F1 и ROC AUC для положительного класса `Occupancy = 1`.

In [5]:
def evaluate(name, X, y):
    prediction = model.predict(X)
    probability = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()

    return {
        "dataset": name,
        "n": len(y),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": accuracy_score(y, prediction),
        "precision": precision_score(y, prediction),
        "recall": recall_score(y, prediction),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y, prediction),
        "roc_auc": roc_auc_score(y, probability),
    }

metrics = pd.DataFrame(
    [
        evaluate("test_1", X_test_1, y_test_1),
        evaluate("test_2", X_test_2, y_test_2),
    ]
).set_index("dataset")

metrics

,n,TN,FP,FN,TP,accuracy,precision,recall,specificity,f1,roc_auc
dataset,,,,,,,,,,,
test_1,2665,1643,50,69,903,0.955347,0.947534,0.929012,0.970467,0.938182,0.985756
test_2,9752,7529,174,14,2035,0.980722,0.921231,0.993167,0.977411,0.955848,0.996762


## 5. Сравнение с опубликованным результатом

В сопровождающем статью авторском R-скрипте для этой спецификации указаны accuracy 0.9553 на первом тесте и 0.9806 на втором. Сравним их с нашим запуском; разность выражена в процентных пунктах.

In [6]:
article_accuracy = pd.Series(
    {"test_1": 0.9553, "test_2": 0.9806},
    name="article_accuracy",
)

comparison = pd.concat([article_accuracy, metrics["accuracy"].rename("our_accuracy")], axis=1)
comparison["difference_pp"] = (
    comparison["our_accuracy"] - comparison["article_accuracy"]
) * 100
comparison["our_correct"] = [
    int(metrics.loc["test_1", "TN"] + metrics.loc["test_1", "TP"]),
    int(metrics.loc["test_2", "TN"] + metrics.loc["test_2", "TP"]),
]
comparison["article_correct_inferred"] = np.rint(
    comparison["article_accuracy"] * metrics["n"]
).astype(int)
comparison["difference_correct"] = (
    comparison["our_correct"] - comparison["article_correct_inferred"]
)
comparison["same_after_4_decimals"] = (
    comparison["our_accuracy"].round(4) == comparison["article_accuracy"]
)
comparison

,article_accuracy,our_accuracy,difference_pp,our_correct,article_correct_inferred,difference_correct,same_after_4_decimals
test_1,0.955300,0.955347,0.004709,2546,2546,0,True
test_2,0.980600,0.980722,0.012190,9564,9563,1,False


In [7]:
importance = pd.Series(
    model.feature_importances_, index=MODEL_FEATURES, name="importance"
).sort_values(ascending=False)
importance.to_frame()

,importance
Light,0.521351
CO2,0.220781
NSM,0.092599
Temperature,0.091816
WeekStatus,0.032575
HumidityRatio,0.022725
Humidity,0.018153


## 6. Вывод

Метрики воспроизводятся практически полностью. На первом тесте 2 546 правильных ответов из 2 665 дают accuracy 0.955347, которая округляется до опубликованных 0.9553. На втором тесте опубликованные 0.9806 совместимы с 9 563 правильными ответами, а у нас их 9 564, то есть наиболее вероятное расхождение составляет ровно один объект из 9 752.

Остаточное отличие ожидаемо и имеет конкретные технические причины:

1. Авторы обучали лес в R (`caret` + `randomForest`), а здесь используется `scikit-learn`. При одинаковом seed генераторы случайных чисел, конкретные бутстрэп-выборки и разрешение равных разбиений между реализациями не обязаны совпадать.
2. Опубликованные accuracy округлены до четырёх знаков. На втором тесте один объект меняет accuracy на `1 / 9752 = 0.000103`, что совпадает с разницей между наиболее вероятным авторским числом правильных ответов и нашим.
3. В R `WeekStatus` является категориальным фактором, а здесь это бинарный признак 0/1. Для дерева это эквивалентное разбиение, но внутреннее представление данных всё равно различается.

Следовательно, эксперимент можно считать успешно воспроизведённым: расхождение не указывает на изменение качества модели и объясняется различием реализаций и округлением.